# EDA Notebook 4 — Discussion Posts Bronze
**Source**: `mini_project_grp6.bronze.discussion_bronze`  
**Format**: NDJSON | ~100K posts/day | Nested threads, parent_post_id  
**Goal**: Thread structure depth, instructor vs student ratios, engagement quality

In [0]:
from pyspark.sql import functions as F

TABLE = "mini_project_grp6.bronze.discussion_bronze"
df    = spark.table(TABLE)
total = df.count()

print(f"Total rows: {total:,}")
df.printSchema()

In [0]:
# ============================================================
# CELL 2 — Null Audit
# ============================================================

null_df = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).toPandas().T.reset_index()
null_df.columns = ["column", "null_count"]
null_df["null_pct"] = (null_df["null_count"] / total * 100).round(2)
print(null_df.sort_values("null_count", ascending=False).to_string(index=False))

In [0]:
# ============================================================
# CELL 3 — Top-Level vs Reply Posts
# ============================================================
# parent_post_id IS NULL = top-level; NOT NULL = reply

display(
    df.withColumn(
        "post_type",
        F.when(F.col("parent_post_id").isNull(), "Top-level")
         .otherwise("Reply")
    )
    .groupBy("post_type")
    .agg(
        F.count("*").alias("count"),
        F.round(F.count("*") / total * 100, 2).alias("pct")
    )
)

In [0]:
# ============================================================
# CELL 4 — Instructor vs Student Posts
# ============================================================

display(
    df.groupBy("is_instructor_post")
      .agg(
          F.count("*").alias("count"),
          F.round(F.avg("like_count"), 2).alias("avg_likes"),
          F.round(F.avg("reply_count"), 2).alias("avg_replies"),
          F.round(F.avg("content_length_chars"), 2).alias("avg_content_len")
      )
)

In [0]:
# ============================================================
# CELL 5 — Content Length Distribution
# ============================================================

print("Content length (chars):")
df.select("content_length_chars").describe().show()

df.selectExpr(
    "percentile_approx(content_length_chars, 0.25) as p25",
    "percentile_approx(content_length_chars, 0.50) as p50",
    "percentile_approx(content_length_chars, 0.75) as p75",
    "percentile_approx(content_length_chars, 0.95) as p95"
).show()

# Flag suspiciously short posts (< 20 chars)
short_posts = df.filter(F.col("content_length_chars") < 20).count()
print(f"\nPosts with < 20 chars: {short_posts:,}")

In [0]:
# ============================================================
# CELL 6 — Thread Engagement Analysis
# ============================================================
# Which threads have the most activity?

display(
    df.groupBy("thread_id")
      .agg(
          F.count("*").alias("post_count"),
          F.sum("like_count").alias("total_likes"),
          F.sum(F.col("is_instructor_post").cast("int")).alias("instructor_posts"),
          F.countDistinct("author_student_id").alias("unique_contributors")
      )
      .orderBy("post_count", ascending=False)
      .limit(20)
)

In [0]:
# ============================================================
# CELL 7 — Self-referencing parent_post_id check
# ============================================================
# Data anomaly: post_id == parent_post_id (self-reply)

self_ref = df.filter(F.col("post_id") == F.col("parent_post_id"))
print(f"Self-referencing posts: {self_ref.count():,}")
display(self_ref.select("post_id", "parent_post_id", "author_student_id").limit(10))

In [0]:
# ============================================================
# CELL 8 — Time Coverage & Daily Volume
# ============================================================

df.selectExpr(
    "min(post_ts) as earliest",
    "max(post_ts) as latest"
).show(truncate=False)

display(
    df.withColumn("post_date", F.to_date("post_ts"))
      .groupBy("post_date")
      .count()
      .orderBy("post_date")
)